# 02 · Flujo de pacientes (process mining)
Etapas: **Ingreso urgencias (triage)** → **Atención en urgencias** (una fila por área/traslado) → **Hospitalización**.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src import datos, flow
from src.config import FIGURES, DATA_INTERIM, DATA_PROCESSED, ETAPAS
pd.set_option("display.max_columns", 50); pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid", context="notebook")
def guardar(nombre):
    plt.tight_layout(); plt.savefig(FIGURES / f"{nombre}.png", dpi=150, bbox_inches="tight"); plt.show()


In [ ]:
et = datos.etapas(); ep = datos.episodios(et)
st = flow.stage_stats(et); st

In [ ]:
tr = (et.groupby(["etapa_anterior", "etapa"])["espera_previa_h"]
        .agg(n="size", espera_p50_h="median", espera_p90_h=lambda x: x.quantile(.9)).reset_index()
        .sort_values("n", ascending=False))
tr["pct"] = tr.n / tr.n.sum() * 100
tr

In [ ]:
seq = ep["secuencia"].str.split(">").map(lambda s: " → ".join(ETAPAS[int(x)] for x in s))
# compacta repeticiones de 'Atención en urgencias' (traslados entre áreas)
import re
var = seq.map(lambda s: re.sub(r"(Atención en urgencias)( → Atención en urgencias)+", r"\1 (×n)", s))
v = var.value_counts().rename_axis("variante").reset_index(name="n")
v["pct"] = v.n / v.n.sum() * 100; v["pct_acum"] = v.pct.cumsum()
v.head(12)

### ¿Dónde se va el tiempo? Composición del episodio

In [ ]:
tipo = ep[ep.los_atipica == 0].groupby("tipo_detalle").agg(
    episodios=("episodio_id", "size"), estancia_p50_h=("los_h", "median"), estancia_p90_h=("los_h", lambda x: x.quantile(.9)),
    h_triage_p50=("h_triage", "median"), h_atencion_p50=("h_atencion_urg", "median"),
    h_hasta_hosp_p50=("h_hasta_hosp", "median"), h_hasta_hosp_p90=("h_hasta_hosp", lambda x: x.quantile(.9)),
    areas_urg_media=("n_areas_urg", "mean"))
tipo

In [ ]:
hosp = ep[ep.tipo_episodio == "Urgencias → Hospitalización"]
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(hosp.h_hasta_hosp.clip(upper=96), bins=96, ax=ax[0])
ax[0].axvline(hosp.h_hasta_hosp.median(), color="red", ls="--"); ax[0].set_title("Horas desde llegada a urgencias hasta hospitalización")
hosp.groupby(hosp.inicio.dt.to_period("Q")).h_hasta_hosp.median().plot(marker="o", ax=ax[1]); ax[1].set_title("Mediana de espera para hospitalización por trimestre (h)")
guardar("02_espera_hospitalizacion")

### Espera entre etapas: huecos > 0 indican pacientes "en tránsito" sin etapa registrada

In [ ]:
gap = et[et.etapa_anterior != "INICIO"].copy()
gap["tipo_gap"] = pd.cut(gap.espera_previa_h, [-1e9, -0.001, 0.001, 1, 6], labels=["superpuesta", "contigua", "<1h", "1-6h"])
pd.crosstab([gap.etapa_anterior, gap.etapa], gap.tipo_gap, normalize="index").mul(100).round(1)

### Mapa del proceso (pm4py, muestra de 50 000 episodios)

In [ ]:
import pm4py
muestra = et[et.episodio_id.isin(ep.episodio_id.sample(50000, random_state=42))]
log = muestra.rename(columns={"episodio_id": "case:concept:name", "etapa": "concept:name", "inicio": "time:timestamp"})
log["case:concept:name"] = log["case:concept:name"].astype(str)
dfg, sa, ea = pm4py.discover_dfg(log[["case:concept:name", "concept:name", "time:timestamp"]])
pd.Series(dfg).sort_values(ascending=False).rename("frecuencia").to_frame()

In [ ]:
try:
    pm4py.save_vis_dfg(dfg, sa, ea, str(FIGURES / "02_mapa_proceso.png"))
    print("Mapa guardado en reports/figures/02_mapa_proceso.png")
except Exception as e:
    print("Graphviz no disponible para dibujar el mapa:", e)

In [ ]:
st.reset_index().to_parquet(DATA_INTERIM / "stage_stats.parquet"); tr.to_parquet(DATA_INTERIM / "transiciones.parquet"); v.to_parquet(DATA_INTERIM / "variantes.parquet")